# Tests de integración de `filter_trips()`

Notebook orientado a verificar el comportamiento end-to-end de OP-05 `Filter trips`,
considerando resultado tabular, resultado operacional y trazabilidad.

## Sección 0. Preparación

Esta sección deja lista la infraestructura mínima del notebook:
- imports generales,
- imports del módulo,
- helpers de testing reutilizables,
- y configuración básica de display.

### 0.1 Imports generales

In [2]:
from copy import deepcopy

import pandas as pd
import h3

### 0.2 Imports del módulo 

In [3]:
from pylondrina.datasets import TripDataset
from pylondrina.errors import FilterError
from pylondrina.schema import DomainSpec, FieldSpec, TripSchema, TripSchemaEffective
from pylondrina.transforms.filtering import FilterOptions, TimeFilter, filter_trips

### 0.3 Helpers de testing reutilizables

In [6]:
def h3_from_latlon(lat: float, lon: float, res: int = 8) -> str:
    if hasattr(h3, "latlng_to_cell"):
        return h3.latlng_to_cell(lat, lon, res)
    return h3.geo_to_h3(lat, lon, res)


def get_last_event(trips: TripDataset) -> dict:
    events = trips.metadata.get("events", [])
    assert isinstance(events, list)
    assert len(events) >= 1
    return events[-1]


def assert_codes_present(report, expected_codes: list[str]) -> None:
    observed = [issue.code for issue in report.issues]
    for code in expected_codes:
        assert code in observed, f"No apareció {code!r}. Observados={observed}"


def assert_input_not_mutated(original: TripDataset, data_before: pd.DataFrame, events_before: list, validated_before: bool) -> None:
    pd.testing.assert_frame_equal(original.data, data_before)
    assert original.metadata["events"] == events_before
    assert original.metadata["is_validated"] == validated_before

### 0.5 Schema y dataframe base ricos

In [7]:
def make_filter_integration_schema() -> TripSchema:
    fields = {
        "movement_id": FieldSpec(name="movement_id", dtype="string", required=True),
        "user_id": FieldSpec(name="user_id", dtype="string", required=True),
        "trip_id": FieldSpec(name="trip_id", dtype="string", required=True),
        "movement_seq": FieldSpec(name="movement_seq", dtype="int", required=True),
        "origin_longitude": FieldSpec(name="origin_longitude", dtype="float", required=True),
        "origin_latitude": FieldSpec(name="origin_latitude", dtype="float", required=True),
        "destination_longitude": FieldSpec(name="destination_longitude", dtype="float", required=True),
        "destination_latitude": FieldSpec(name="destination_latitude", dtype="float", required=True),
        "origin_h3_index": FieldSpec(name="origin_h3_index", dtype="string", required=True),
        "destination_h3_index": FieldSpec(name="destination_h3_index", dtype="string", required=True),
        "origin_time_utc": FieldSpec(name="origin_time_utc", dtype="datetime", required=True),
        "destination_time_utc": FieldSpec(name="destination_time_utc", dtype="datetime", required=True),
        "origin_time_local_hhmm": FieldSpec(name="origin_time_local_hhmm", dtype="string"),
        "destination_time_local_hhmm": FieldSpec(name="destination_time_local_hhmm", dtype="string"),
        "origin_municipality": FieldSpec(name="origin_municipality", dtype="string"),
        "destination_municipality": FieldSpec(name="destination_municipality", dtype="string"),
        "mode": FieldSpec(
            name="mode",
            dtype="categorical",
            domain=DomainSpec(
                values=["walk", "bicycle", "car", "bus", "metro", "other", "scooter"],
                extendable=True,
            ),
        ),
        "purpose": FieldSpec(
            name="purpose",
            dtype="categorical",
            domain=DomainSpec(
                values=["home", "work", "study", "shopping", "leisure", "other"],
                extendable=True,
            ),
        ),
        "day_type": FieldSpec(
            name="day_type",
            dtype="categorical",
            domain=DomainSpec(values=["weekday", "weekend", "holiday"], extendable=True),
        ),
        "time_period": FieldSpec(
            name="time_period",
            dtype="categorical",
            domain=DomainSpec(values=["night", "morning", "midday", "afternoon", "evening"], extendable=True),
        ),
        "user_gender": FieldSpec(
            name="user_gender",
            dtype="categorical",
            domain=DomainSpec(values=["female", "male", "other", "unknown"], extendable=True),
        ),
        "income_quintile": FieldSpec(
            name="income_quintile",
            dtype="categorical",
            domain=DomainSpec(values=["1", "2", "3", "4", "5", "unknown"], extendable=True),
        ),
        "trip_weight": FieldSpec(name="trip_weight", dtype="float"),
        "distance_km": FieldSpec(name="distance_km", dtype="float"),
        "is_peak": FieldSpec(name="is_peak", dtype="bool"),
        "mode_sequence": FieldSpec(name="mode_sequence", dtype="string"),
    }

    return TripSchema(
        version="test-filter-1.0",
        fields=fields,
        required=[
            "movement_id",
            "user_id",
            "trip_id",
            "movement_seq",
            "origin_longitude",
            "origin_latitude",
            "destination_longitude",
            "destination_latitude",
            "origin_h3_index",
            "destination_h3_index",
            "origin_time_utc",
            "destination_time_utc",
        ],
    )


def make_filter_integration_dataframe() -> pd.DataFrame:
    rows = [
        # m0: dentro-dentro, AM, bus/work
        dict(movement_id="m0", user_id="u0", trip_id="t0", movement_seq=0,
             mode="bus", purpose="work", day_type="weekday", time_period="morning",
             user_gender="female", income_quintile="3", trip_weight=1.2, distance_km=5.0,
             is_peak=True, mode_sequence="bus",
             origin_municipality="Santiago", destination_municipality="Providencia",
             origin_longitude=-70.66, origin_latitude=-33.45,
             destination_longitude=-70.65, destination_latitude=-33.44,
             origin_time_utc="2026-01-01T07:10:00Z", destination_time_utc="2026-01-01T07:40:00Z"),
        # m1: dentro-dentro, AM, metro/study
        dict(movement_id="m1", user_id="u1", trip_id="t1", movement_seq=0,
             mode="metro", purpose="study", day_type="weekday", time_period="morning",
             user_gender="male", income_quintile="2", trip_weight=0.8, distance_km=12.5,
             is_peak=True, mode_sequence="metro",
             origin_municipality="Ñuñoa", destination_municipality="Santiago",
             origin_longitude=-70.64, origin_latitude=-33.46,
             destination_longitude=-70.63, destination_latitude=-33.45,
             origin_time_utc="2026-01-01T08:00:00Z", destination_time_utc="2026-01-01T08:35:00Z"),
        # m2: fuera-fuera, AM tardío, car/work
        dict(movement_id="m2", user_id="u2", trip_id="t2", movement_seq=0,
             mode="car", purpose="work", day_type="weekday", time_period="morning",
             user_gender="female", income_quintile="5", trip_weight=1.5, distance_km=1.2,
             is_peak=False, mode_sequence="car",
             origin_municipality="Puente Alto", destination_municipality="Puente Alto",
             origin_longitude=-70.80, origin_latitude=-33.60,
             destination_longitude=-70.79, destination_latitude=-33.59,
             origin_time_utc="2026-01-01T09:30:00Z", destination_time_utc="2026-01-01T10:00:00Z"),
        # m3: origen dentro, destino fuera
        dict(movement_id="m3", user_id="u3", trip_id="t3", movement_seq=0,
             mode="walk", purpose="leisure", day_type="weekday", time_period="morning",
             user_gender="male", income_quintile="1", trip_weight=0.7, distance_km=0.4,
             is_peak=False, mode_sequence="walk",
             origin_municipality="Santiago", destination_municipality="Quilicura",
             origin_longitude=-70.66, origin_latitude=-33.45,
             destination_longitude=-70.81, destination_latitude=-33.61,
             origin_time_utc="2026-01-01T10:00:00Z", destination_time_utc="2026-01-01T10:20:00Z"),
        # m4: fuera-fuera, leisure
        dict(movement_id="m4", user_id="u4", trip_id="t4", movement_seq=0,
             mode="bus", purpose="leisure", day_type="weekend", time_period="midday",
             user_gender="female", income_quintile="4", trip_weight=1.1, distance_km=25.0,
             is_peak=True, mode_sequence="bus",
             origin_municipality="Maipú", destination_municipality="Maipú",
             origin_longitude=-70.90, origin_latitude=-33.70,
             destination_longitude=-70.91, destination_latitude=-33.71,
             origin_time_utc="2026-01-01T11:00:00Z", destination_time_utc="2026-01-01T11:30:00Z"),
        # m5: dentro-dentro, metro/work
        dict(movement_id="m5", user_id="u5", trip_id="t5", movement_seq=0,
             mode="metro", purpose="work", day_type="weekday", time_period="morning",
             user_gender="other", income_quintile="2", trip_weight=1.0, distance_km=8.2,
             is_peak=True, mode_sequence="metro",
             origin_municipality="Providencia", destination_municipality="Santiago",
             origin_longitude=-70.62, origin_latitude=-33.43,
             destination_longitude=-70.63, destination_latitude=-33.44,
             origin_time_utc="2026-01-01T07:50:00Z", destination_time_utc="2026-01-01T08:10:00Z"),
        # m6: dentro-dentro, bicycle/work
        dict(movement_id="m6", user_id="u6", trip_id="t6", movement_seq=0,
             mode="bicycle", purpose="work", day_type="weekday", time_period="morning",
             user_gender="male", income_quintile="1", trip_weight=0.9, distance_km=3.0,
             is_peak=True, mode_sequence="bicycle",
             origin_municipality="Providencia", destination_municipality="Ñuñoa",
             origin_longitude=-70.61, origin_latitude=-33.42,
             destination_longitude=-70.62, destination_latitude=-33.43,
             origin_time_utc="2026-01-01T08:20:00Z", destination_time_utc="2026-01-01T08:50:00Z"),
        # m7: dentro-dentro, demasiado temprano
        dict(movement_id="m7", user_id="u7", trip_id="t7", movement_seq=0,
             mode="bus", purpose="shopping", day_type="weekday", time_period="morning",
             user_gender="female", income_quintile="2", trip_weight=0.6, distance_km=6.0,
             is_peak=False, mode_sequence="bus",
             origin_municipality="Santiago", destination_municipality="Santiago",
             origin_longitude=-70.67, origin_latitude=-33.46,
             destination_longitude=-70.66, destination_latitude=-33.45,
             origin_time_utc="2026-01-01T06:30:00Z", destination_time_utc="2026-01-01T06:50:00Z"),
        # m8: origen fuera, destino dentro
        dict(movement_id="m8", user_id="u8", trip_id="t8", movement_seq=0,
             mode="car", purpose="work", day_type="weekday", time_period="morning",
             user_gender="female", income_quintile="3", trip_weight=1.3, distance_km=14.0,
             is_peak=True, mode_sequence="car",
             origin_municipality="Pudahuel", destination_municipality="Providencia",
             origin_longitude=-70.82, origin_latitude=-33.61,
             destination_longitude=-70.64, destination_latitude=-33.45,
             origin_time_utc="2026-01-01T08:10:00Z", destination_time_utc="2026-01-01T08:40:00Z"),
        # m9: dentro-dentro, overlaps al borde
        dict(movement_id="m9", user_id="u9", trip_id="t9", movement_seq=0,
             mode="metro", purpose="work", day_type="weekday", time_period="morning",
             user_gender="male", income_quintile="4", trip_weight=1.4, distance_km=10.5,
             is_peak=True, mode_sequence="metro",
             origin_municipality="Ñuñoa", destination_municipality="Providencia",
             origin_longitude=-70.63, origin_latitude=-33.44,
             destination_longitude=-70.64, destination_latitude=-33.45,
             origin_time_utc="2026-01-01T08:45:00Z", destination_time_utc="2026-01-01T09:15:00Z"),
        # m10: dentro-dentro, bus/work, cruza apenas fin del rango
        dict(movement_id="m10", user_id="u10", trip_id="t10", movement_seq=0,
             mode="bus", purpose="work", day_type="weekday", time_period="morning",
             user_gender="female", income_quintile="3", trip_weight=1.0, distance_km=7.7,
             is_peak=True, mode_sequence="bus",
             origin_municipality="Santiago", destination_municipality="Providencia",
             origin_longitude=-70.65, origin_latitude=-33.44,
             destination_longitude=-70.64, destination_latitude=-33.43,
             origin_time_utc="2026-01-01T08:59:00Z", destination_time_utc="2026-01-01T09:05:00Z"),
        # m11: dentro-dentro, tarde, walk/study
        dict(movement_id="m11", user_id="u11", trip_id="t11", movement_seq=0,
             mode="walk", purpose="study", day_type="holiday", time_period="afternoon",
             user_gender="unknown", income_quintile="1", trip_weight=0.5, distance_km=1.0,
             is_peak=False, mode_sequence="walk",
             origin_municipality="Providencia", destination_municipality="Ñuñoa",
             origin_longitude=-70.68, origin_latitude=-33.47,
             destination_longitude=-70.67, destination_latitude=-33.46,
             origin_time_utc="2026-01-01T14:00:00Z", destination_time_utc="2026-01-01T14:30:00Z"),
    ]

    df = pd.DataFrame(rows)
    df["origin_time_utc"] = pd.to_datetime(df["origin_time_utc"], utc=True)
    df["destination_time_utc"] = pd.to_datetime(df["destination_time_utc"], utc=True)

    df["origin_time_local_hhmm"] = df["origin_time_utc"].dt.strftime("%H:%M")
    df["destination_time_local_hhmm"] = df["destination_time_utc"].dt.strftime("%H:%M")

    df["origin_h3_index"] = [
        h3_from_latlon(lat, lon, 8)
        for lat, lon in zip(df["origin_latitude"], df["origin_longitude"])
    ]
    df["destination_h3_index"] = [
        h3_from_latlon(lat, lon, 8)
        for lat, lon in zip(df["destination_latitude"], df["destination_longitude"])
    ]
    return df

### 0.6 Fixtures tripdataset_with_time_space_fields, tripdataset_canonical_small, tripdataset_validated_small

In [8]:
def _build_schema_effective(df: pd.DataFrame) -> TripSchemaEffective:
    dtype_effective = {
        "movement_id": "string",
        "user_id": "string",
        "trip_id": "string",
        "movement_seq": "int",
        "origin_longitude": "float",
        "origin_latitude": "float",
        "destination_longitude": "float",
        "destination_latitude": "float",
        "origin_h3_index": "string",
        "destination_h3_index": "string",
        "origin_time_utc": "datetime",
        "destination_time_utc": "datetime",
        "origin_time_local_hhmm": "string",
        "destination_time_local_hhmm": "string",
        "origin_municipality": "string",
        "destination_municipality": "string",
        "mode": "categorical",
        "purpose": "categorical",
        "day_type": "categorical",
        "time_period": "categorical",
        "user_gender": "categorical",
        "income_quintile": "categorical",
        "trip_weight": "float",
        "distance_km": "float",
        "is_peak": "bool",
        "mode_sequence": "string",
    }
    return TripSchemaEffective(
        dtype_effective={k: v for k, v in dtype_effective.items() if k in df.columns},
        domains_effective={
            "mode": {"values": ["walk", "bicycle", "car", "bus", "metro", "other", "scooter"]},
            "purpose": {"values": ["home", "work", "study", "shopping", "leisure", "other"]},
            "day_type": {"values": ["weekday", "weekend", "holiday"]},
            "time_period": {"values": ["night", "morning", "midday", "afternoon", "evening"]},
            "user_gender": {"values": ["female", "male", "other", "unknown"]},
            "income_quintile": {"values": ["1", "2", "3", "4", "5", "unknown"]},
        },
        temporal={"tier": "tier_1"},
        fields_effective=list(df.columns),
    )


def tripdataset_with_time_space_fields(*, validated: bool = False, temporal_tier: str = "tier_1") -> TripDataset:
    df = make_filter_integration_dataframe()
    schema = make_filter_integration_schema()

    metadata = {
        "dataset_id": "ds_filter_integration_rich",
        "is_validated": bool(validated),
        "temporal": {
            "tier": temporal_tier,
            "fields_present": (
                ["origin_time_utc", "destination_time_utc"]
                if temporal_tier == "tier_1"
                else ["origin_time_local_hhmm", "destination_time_local_hhmm"]
            ),
        },
        "h3": {
            "resolution": 8,
            "derived_fields": ["origin_h3_index", "destination_h3_index"],
        },
        "schema": {"schema_version": schema.version},
        "domains_effective": {
            "mode": {"values": ["walk", "bicycle", "car", "bus", "metro", "other", "scooter"]},
            "purpose": {"values": ["home", "work", "study", "shopping", "leisure", "other"]},
        },
        "events": [
            {
                "op": "import_trips",
                "ts_utc": "2026-04-03T12:00:00Z",
                "parameters": {"source_name": "synthetic_filter_integration"},
                "summary": {"rows_in": len(df), "rows_out": len(df)},
                "issues_summary": {"counts": {"info": 0, "warning": 0, "error": 0}, "top_codes": []},
            }
        ],
    }

    if validated:
        metadata["events"].append(
            {
                "op": "validate_trips",
                "ts_utc": "2026-04-03T12:10:00Z",
                "parameters": {"strict": False},
                "summary": {"ok": True, "n_rows": len(df), "n_errors": 0},
                "issues_summary": {"counts": {"info": 0, "warning": 0, "error": 0}, "top_codes": []},
            }
        )

    schema_effective = _build_schema_effective(df)
    schema_effective.temporal = {"tier": temporal_tier}

    return TripDataset(
        data=df,
        schema=schema,
        schema_version=schema.version,
        provenance={"source": {"name": "synthetic_filter_integration"}},
        field_correspondence={},
        value_correspondence={},
        metadata=metadata,
        schema_effective=schema_effective,
    )


def tripdataset_canonical_small(*, validated: bool = False) -> TripDataset:
    trips = tripdataset_with_time_space_fields(validated=validated, temporal_tier="tier_1")
    trips.data = trips.data.iloc[:8].reset_index(drop=True)
    trips.metadata["dataset_id"] = "ds_filter_integration_small"
    trips.metadata["events"][0]["summary"] = {
        "rows_in": len(trips.data),
        "rows_out": len(trips.data),
    }
    if validated and len(trips.metadata["events"]) > 1:
        trips.metadata["events"][1]["summary"]["n_rows"] = len(trips.data)
    trips.schema_effective.fields_effective = list(trips.data.columns)
    return trips


def tripdataset_validated_small() -> TripDataset:
    return tripdataset_canonical_small(validated=True)

### 0.4 Configuración de display

In [5]:
pd.set_option("display.max_columns", 120)
pd.set_option("display.width", 200)
pd.set_option("display.max_colwidth", 120)

## Sección 1. Integration tests de filter_trips

### Test 1 - caso principal correcto: combinación where + time + bbox

Qué prueba: el camino feliz principal sobre un fixture rico, verificando output tabular, OperationReport, summary, parameters, evento y preservación de is_validated.

In [10]:
trips = tripdataset_with_time_space_fields(validated=True)
data_before = trips.data.copy(deep=True)
events_before = deepcopy(trips.metadata["events"])
validated_before = trips.metadata["is_validated"]

options = FilterOptions(
    where={
        "mode": ["bus", "metro"],
        "purpose": ["work", "study"],
    },
    time=TimeFilter(
        start="2026-01-01T07:00:00Z",
        end="2026-01-01T09:00:00Z",
        predicate="overlaps",
    ),
    bbox=(-70.70, -33.50, -70.60, -33.40),
)

filtered, report = filter_trips(
    trips,
    options=options,
    max_issues=50,
    sample_rows_per_issue=5,
)

assert filtered.data["movement_id"].tolist() == ["m0", "m1", "m5", "m9", "m10"]

assert report.ok is True
assert report.summary["rows_in"] == 12
assert report.summary["rows_out"] == 5
assert report.summary["dropped_total"] == 7
assert report.summary["filters_requested"] == ["where", "time", "bbox"]
assert report.summary["filters_applied"] == ["where", "time", "bbox"]
assert report.summary["filters_omitted"] == []

assert report.parameters["max_issues"] == 50
assert report.parameters["sample_rows_per_issue"] == 5
assert report.parameters["origin_h3_field"] == "origin_h3_index"
assert report.parameters["destination_h3_field"] == "destination_h3_index"

assert_codes_present(report, [
    "FLT.INFO.WHERE_APPLIED",
    "FLT.INFO.TIME_APPLIED",
    "FLT.INFO.BBOX_APPLIED",
])

event = get_last_event(filtered)
assert event["op"] == "filter_trips"
assert event["summary"] == report.summary
assert event["parameters"] == report.parameters
assert "issues_summary" in event

assert filtered.metadata["is_validated"] is validated_before
assert_input_not_mutated(trips, data_before, events_before, validated_before)

display(trips.data)
display(filtered.data)
display(report.issues)

,movement_id,user_id,trip_id,movement_seq,mode,purpose,day_type,time_period,user_gender,income_quintile,trip_weight,distance_km,is_peak,mode_sequence,origin_municipality,destination_municipality,origin_longitude,origin_latitude,destination_longitude,destination_latitude,origin_time_utc,destination_time_utc,origin_time_local_hhmm,destination_time_local_hhmm,origin_h3_index,destination_h3_index
0,m0,u0,t0,0,bus,work,weekday,morning,female,3,1.2,5.0,True,bus,Santiago,Providencia,-70.66,-33.45,-70.65,-33.44,2026-01-01 07:10:00+00:00,2026-01-01 07:40:00+00:00,07:10,07:40,88b2c55417fffff,88b2c5541bfffff
1,m1,u1,t1,0,metro,study,weekday,morning,male,2,0.8,12.5,True,metro,Ñuñoa,Santiago,-70.64,-33.46,-70.63,-33.45,2026-01-01 08:00:00+00:00,2026-01-01 08:35:00+00:00,08:00,08:35,88b2c5548dfffff,88b2c554c1fffff
2,m2,u2,t2,0,car,work,weekday,morning,female,5,1.5,1.2,False,car,Puente Alto,Puente Alto,-70.80,-33.60,-70.79,-33.59,2026-01-01 09:30:00+00:00,2026-01-01 10:00:00+00:00,09:30,10:00,88b2c54e11fffff,88b2c54525fffff
3,m3,u3,t3,0,walk,leisure,weekday,morning,male,1,0.7,0.4,False,walk,Santiago,Quilicura,-70.66,-33.45,-70.81,-33.61,2026-01-01 10:00:00+00:00,2026-01-01 10:20:00+00:00,10:00,10:20,88b2c55417fffff,88b2c54e3bfffff
4,m4,u4,t4,0,bus,leisure,weekend,midday,female,4,1.1,25.0,True,bus,Maipú,Maipú,-70.90,-33.70,-70.91,-33.71,2026-01-01 11:00:00+00:00,2026-01-01 11:30:00+00:00,11:00,11:30,88b2c54da1fffff,88b2c0b29bfffff
5,m5,u5,t5,0,metro,work,weekday,morning,other,2,1.0,8.2,True,metro,Providencia,Santiago,-70.62,-33.43,-70.63,-33.44,2026-01-01 07:50:00+00:00,2026-01-01 08:10:00+00:00,07:50,08:10,88b2c556bdfffff,88b2c554c9fffff
6,m6,u6,t6,0,bicycle,work,weekday,morning,male,1,0.9,3.0,True,bicycle,Providencia,Ñuñoa,-70.61,-33.42,-70.62,-33.43,2026-01-01 08:20:00+00:00,2026-01-01 08:50:00+00:00,08:20,08:50,88b2c55681fffff,88b2c556bdfffff
7,m7,u7,t7,0,bus,shopping,weekday,morning,female,2,0.6,6.0,False,bus,Santiago,Santiago,-70.67,-33.46,-70.66,-33.45,2026-01-01 06:30:00+00:00,2026-01-01 06:50:00+00:00,06:30,06:50,88b2c554e5fffff,88b2c55417fffff
8,m8,u8,t8,0,car,work,weekday,morning,female,3,1.3,14.0,True,car,Pudahuel,Providencia,-70.82,-33.61,-70.64,-33.45,2026-01-01 08:10:00+00:00,2026-01-01 08:40:00+00:00,08:10,08:40,88b2c54e31fffff,88b2c554c5fffff
9,m9,u9,t9,0,metro,work,weekday,morning,male,4,1.4,10.5,True,metro,Ñuñoa,Providencia,-70.63,-33.44,-70.64,-33.45,2026-01-01 08:45:00+00:00,2026-01-01 09:15:00+00:00,08:45,09:15,88b2c554c9fffff,88b2c554c5fffff


,movement_id,user_id,trip_id,movement_seq,mode,purpose,day_type,time_period,user_gender,income_quintile,trip_weight,distance_km,is_peak,mode_sequence,origin_municipality,destination_municipality,origin_longitude,origin_latitude,destination_longitude,destination_latitude,origin_time_utc,destination_time_utc,origin_time_local_hhmm,destination_time_local_hhmm,origin_h3_index,destination_h3_index
0,m0,u0,t0,0,bus,work,weekday,morning,female,3,1.2,5.0,True,bus,Santiago,Providencia,-70.66,-33.45,-70.65,-33.44,2026-01-01 07:10:00+00:00,2026-01-01 07:40:00+00:00,07:10,07:40,88b2c55417fffff,88b2c5541bfffff
1,m1,u1,t1,0,metro,study,weekday,morning,male,2,0.8,12.5,True,metro,Ñuñoa,Santiago,-70.64,-33.46,-70.63,-33.45,2026-01-01 08:00:00+00:00,2026-01-01 08:35:00+00:00,08:00,08:35,88b2c5548dfffff,88b2c554c1fffff
5,m5,u5,t5,0,metro,work,weekday,morning,other,2,1.0,8.2,True,metro,Providencia,Santiago,-70.62,-33.43,-70.63,-33.44,2026-01-01 07:50:00+00:00,2026-01-01 08:10:00+00:00,07:50,08:10,88b2c556bdfffff,88b2c554c9fffff
9,m9,u9,t9,0,metro,work,weekday,morning,male,4,1.4,10.5,True,metro,Ñuñoa,Providencia,-70.63,-33.44,-70.64,-33.45,2026-01-01 08:45:00+00:00,2026-01-01 09:15:00+00:00,08:45,09:15,88b2c554c9fffff,88b2c554c5fffff
10,m10,u10,t10,0,bus,work,weekday,morning,female,3,1.0,7.7,True,bus,Santiago,Providencia,-70.65,-33.44,-70.64,-33.43,2026-01-01 08:59:00+00:00,2026-01-01 09:05:00+00:00,08:59,09:05,88b2c5541bfffff,88b2c556a5fffff


[Issue(level='info', code='FLT.INFO.WHERE_APPLIED', message='Se aplicó la cláusula where y el subconjunto pasó de 12 a 5 filas en este eje.', field=None, source_field=None, row_count=None, details={'check': 'where', 'where_fields': ['mode', 'purpose'], 'rows_in_scope': 12, 'rows_out_scope': 5, 'rows_removed_scope': 7, 'row_indices_sample_removed': [2, 3, 4, 6, 7], 'movement_ids_sample_removed': ['m2', 'm3', 'm4', 'm6', 'm7'], 'values_sample': [{'mode': 'car', 'purpose': 'work'}, {'mode': 'walk', 'purpose': 'leisure'}, {'mode': 'bus', 'purpose': 'leisure'}, {'mode': 'bicycle', 'purpose': 'work'}, {'mode': 'bus', 'purpose': 'shopping'}], 'policy': 'and_between_fields_and_ops'}),
 Issue(level='info', code='FLT.INFO.TIME_APPLIED', message="Se aplicó el filtro temporal ('overlaps') y el subconjunto pasó de 12 a 7 filas en este eje.", field=None, source_field=None, row_count=None, details={'check': 'time', 'rows_in_scope': 12, 'rows_out_scope': 7, 'rows_removed_scope': 5, 'row_indices_sample

### Test 2 - caso principal espacial: spatial_predicate="both" sobre bbox

Qué prueba: que la semántica espacial both funcione bien y excluya viajes donde solo uno de los extremos cae dentro del área.

In [12]:
trips = tripdataset_with_time_space_fields(validated=False)

options = FilterOptions(
    bbox=(-70.70, -33.50, -70.60, -33.40),
    spatial_predicate="both",
)

filtered, report = filter_trips(trips, options=options)

# Quedan solo los viajes con origen y destino dentro del bbox.
assert filtered.data["movement_id"].tolist() == ["m0", "m1", "m5", "m6", "m7", "m9", "m10", "m11"]

assert report.ok is True
assert report.summary["rows_in"] == 12
assert report.summary["rows_out"] == 8
assert report.summary["dropped_total"] == 4
assert report.summary["filters_requested"] == ["bbox"]
assert report.summary["filters_applied"] == ["bbox"]
assert report.summary["filters_omitted"] == []

assert_codes_present(report, ["FLT.INFO.BBOX_APPLIED"])
display(trips.data)
display(filtered.data)
display(report.issues)

,movement_id,user_id,trip_id,movement_seq,mode,purpose,day_type,time_period,user_gender,income_quintile,trip_weight,distance_km,is_peak,mode_sequence,origin_municipality,destination_municipality,origin_longitude,origin_latitude,destination_longitude,destination_latitude,origin_time_utc,destination_time_utc,origin_time_local_hhmm,destination_time_local_hhmm,origin_h3_index,destination_h3_index
0,m0,u0,t0,0,bus,work,weekday,morning,female,3,1.2,5.0,True,bus,Santiago,Providencia,-70.66,-33.45,-70.65,-33.44,2026-01-01 07:10:00+00:00,2026-01-01 07:40:00+00:00,07:10,07:40,88b2c55417fffff,88b2c5541bfffff
1,m1,u1,t1,0,metro,study,weekday,morning,male,2,0.8,12.5,True,metro,Ñuñoa,Santiago,-70.64,-33.46,-70.63,-33.45,2026-01-01 08:00:00+00:00,2026-01-01 08:35:00+00:00,08:00,08:35,88b2c5548dfffff,88b2c554c1fffff
2,m2,u2,t2,0,car,work,weekday,morning,female,5,1.5,1.2,False,car,Puente Alto,Puente Alto,-70.80,-33.60,-70.79,-33.59,2026-01-01 09:30:00+00:00,2026-01-01 10:00:00+00:00,09:30,10:00,88b2c54e11fffff,88b2c54525fffff
3,m3,u3,t3,0,walk,leisure,weekday,morning,male,1,0.7,0.4,False,walk,Santiago,Quilicura,-70.66,-33.45,-70.81,-33.61,2026-01-01 10:00:00+00:00,2026-01-01 10:20:00+00:00,10:00,10:20,88b2c55417fffff,88b2c54e3bfffff
4,m4,u4,t4,0,bus,leisure,weekend,midday,female,4,1.1,25.0,True,bus,Maipú,Maipú,-70.90,-33.70,-70.91,-33.71,2026-01-01 11:00:00+00:00,2026-01-01 11:30:00+00:00,11:00,11:30,88b2c54da1fffff,88b2c0b29bfffff
5,m5,u5,t5,0,metro,work,weekday,morning,other,2,1.0,8.2,True,metro,Providencia,Santiago,-70.62,-33.43,-70.63,-33.44,2026-01-01 07:50:00+00:00,2026-01-01 08:10:00+00:00,07:50,08:10,88b2c556bdfffff,88b2c554c9fffff
6,m6,u6,t6,0,bicycle,work,weekday,morning,male,1,0.9,3.0,True,bicycle,Providencia,Ñuñoa,-70.61,-33.42,-70.62,-33.43,2026-01-01 08:20:00+00:00,2026-01-01 08:50:00+00:00,08:20,08:50,88b2c55681fffff,88b2c556bdfffff
7,m7,u7,t7,0,bus,shopping,weekday,morning,female,2,0.6,6.0,False,bus,Santiago,Santiago,-70.67,-33.46,-70.66,-33.45,2026-01-01 06:30:00+00:00,2026-01-01 06:50:00+00:00,06:30,06:50,88b2c554e5fffff,88b2c55417fffff
8,m8,u8,t8,0,car,work,weekday,morning,female,3,1.3,14.0,True,car,Pudahuel,Providencia,-70.82,-33.61,-70.64,-33.45,2026-01-01 08:10:00+00:00,2026-01-01 08:40:00+00:00,08:10,08:40,88b2c54e31fffff,88b2c554c5fffff
9,m9,u9,t9,0,metro,work,weekday,morning,male,4,1.4,10.5,True,metro,Ñuñoa,Providencia,-70.63,-33.44,-70.64,-33.45,2026-01-01 08:45:00+00:00,2026-01-01 09:15:00+00:00,08:45,09:15,88b2c554c9fffff,88b2c554c5fffff


,movement_id,user_id,trip_id,movement_seq,mode,purpose,day_type,time_period,user_gender,income_quintile,trip_weight,distance_km,is_peak,mode_sequence,origin_municipality,destination_municipality,origin_longitude,origin_latitude,destination_longitude,destination_latitude,origin_time_utc,destination_time_utc,origin_time_local_hhmm,destination_time_local_hhmm,origin_h3_index,destination_h3_index
0,m0,u0,t0,0,bus,work,weekday,morning,female,3,1.2,5.0,True,bus,Santiago,Providencia,-70.66,-33.45,-70.65,-33.44,2026-01-01 07:10:00+00:00,2026-01-01 07:40:00+00:00,07:10,07:40,88b2c55417fffff,88b2c5541bfffff
1,m1,u1,t1,0,metro,study,weekday,morning,male,2,0.8,12.5,True,metro,Ñuñoa,Santiago,-70.64,-33.46,-70.63,-33.45,2026-01-01 08:00:00+00:00,2026-01-01 08:35:00+00:00,08:00,08:35,88b2c5548dfffff,88b2c554c1fffff
5,m5,u5,t5,0,metro,work,weekday,morning,other,2,1.0,8.2,True,metro,Providencia,Santiago,-70.62,-33.43,-70.63,-33.44,2026-01-01 07:50:00+00:00,2026-01-01 08:10:00+00:00,07:50,08:10,88b2c556bdfffff,88b2c554c9fffff
6,m6,u6,t6,0,bicycle,work,weekday,morning,male,1,0.9,3.0,True,bicycle,Providencia,Ñuñoa,-70.61,-33.42,-70.62,-33.43,2026-01-01 08:20:00+00:00,2026-01-01 08:50:00+00:00,08:20,08:50,88b2c55681fffff,88b2c556bdfffff
7,m7,u7,t7,0,bus,shopping,weekday,morning,female,2,0.6,6.0,False,bus,Santiago,Santiago,-70.67,-33.46,-70.66,-33.45,2026-01-01 06:30:00+00:00,2026-01-01 06:50:00+00:00,06:30,06:50,88b2c554e5fffff,88b2c55417fffff
9,m9,u9,t9,0,metro,work,weekday,morning,male,4,1.4,10.5,True,metro,Ñuñoa,Providencia,-70.63,-33.44,-70.64,-33.45,2026-01-01 08:45:00+00:00,2026-01-01 09:15:00+00:00,08:45,09:15,88b2c554c9fffff,88b2c554c5fffff
10,m10,u10,t10,0,bus,work,weekday,morning,female,3,1.0,7.7,True,bus,Santiago,Providencia,-70.65,-33.44,-70.64,-33.43,2026-01-01 08:59:00+00:00,2026-01-01 09:05:00+00:00,08:59,09:05,88b2c5541bfffff,88b2c556a5fffff
11,m11,u11,t11,0,walk,study,holiday,afternoon,unknown,1,0.5,1.0,False,walk,Providencia,Ñuñoa,-70.68,-33.47,-70.67,-33.46,2026-01-01 14:00:00+00:00,2026-01-01 14:30:00+00:00,14:00,14:30,88b2c555d1fffff,88b2c554e5fffff


[Issue(level='info', code='FLT.INFO.BBOX_APPLIED', message='Se aplicó el filtro bbox y el subconjunto pasó de 12 a 8 filas en este eje.', field=None, source_field=None, row_count=None, details={'check': 'spatial', 'rows_in_scope': 12, 'rows_out_scope': 8, 'rows_removed_scope': 4, 'row_indices_sample_removed': [2, 3, 4, 8], 'movement_ids_sample_removed': ['m2', 'm3', 'm4', 'm8'], 'values_sample': [{'origin_longitude': -70.8, 'origin_latitude': -33.6, 'destination_longitude': -70.79, 'destination_latitude': -33.59}, {'origin_longitude': -70.66, 'origin_latitude': -33.45, 'destination_longitude': -70.81, 'destination_latitude': -33.61}, {'origin_longitude': -70.9, 'origin_latitude': -33.7, 'destination_longitude': -70.91, 'destination_latitude': -33.71}, {'origin_longitude': -70.82, 'origin_latitude': -33.61, 'destination_longitude': -70.64, 'destination_latitude': -33.45}], 'policy': 'and_between_spatial_filters', 'bbox': [-70.7, -33.5, -70.6, -33.4], 'spatial_predicate': 'both'})]

### Test 3 - fatal/precondición: bbox inválido aborta sin mutar el input

Qué prueba: abort fatal de configuración antes del pipeline normal, sin side effects sobre el input.

In [14]:
trips = tripdataset_with_time_space_fields(validated=True)
data_before = trips.data.copy(deep=True)
events_before = deepcopy(trips.metadata["events"])
validated_before = trips.metadata["is_validated"]

raised = None
try:
    filter_trips(
        trips,
        options=FilterOptions(
            bbox=(-70.70, -33.50, -70.80, -33.40),  # min_lon > max_lon
        ),
    )
except Exception as e:
    raised = e

assert raised is not None
assert isinstance(raised, ValueError)

assert_input_not_mutated(trips, data_before, events_before, validated_before)
display(raised.issues)

(Issue(level='error', code='FLT.SPATIAL.INVALID_BBOX', message='La geometría bbox no es interpretable como (min_lon, min_lat, max_lon, max_lat).', field=None, source_field=None, row_count=None, details={'check': 'spatial', 'bbox': [-70.7, -33.5, -70.8, -33.4], 'observed': {'min_lon': -70.7, 'min_lat': -33.5, 'max_lon': -70.8, 'max_lat': -33.4}, 'expected': 'tuple/list[float, float, float, float] with min<=max', 'reason': 'invalid_bbox', 'action': 'abort'}),)

### Test 4 - caso degradado con strict=False: Tier 2 omite time pero aplica bbox

Qué prueba: degradación controlada con issue de error retornable, sin abortar, aplicando los otros filtros.

In [16]:
trips = tripdataset_with_time_space_fields(validated=False, temporal_tier="tier_2")
data_before = trips.data.copy(deep=True)
events_before = deepcopy(trips.metadata["events"])
validated_before = trips.metadata["is_validated"]

options = FilterOptions(
    time=TimeFilter(
        start="2026-01-01T07:00:00Z",
        end="2026-01-01T09:00:00Z",
        predicate="overlaps",
    ),
    bbox=(-70.70, -33.50, -70.60, -33.40),
    strict=False,
)

filtered, report = filter_trips(trips, options=options)

# bbox con spatial_predicate por defecto = origin
assert filtered.data["movement_id"].tolist() == ["m0", "m1", "m3", "m5", "m6", "m7", "m9", "m10", "m11"]

assert report.ok is False
assert report.summary["filters_requested"] == ["time", "bbox"]
assert report.summary["filters_applied"] == ["bbox"]
assert report.summary["filters_omitted"] == ["time"]

assert_codes_present(report, [
    "FLT.TIME.UNSUPPORTED_TIER",
    "FLT.INFO.BBOX_APPLIED",
])

event = get_last_event(filtered)
assert event["op"] == "filter_trips"
assert event["summary"] == report.summary
assert filtered.metadata["is_validated"] == validated_before

assert_input_not_mutated(trips, data_before, events_before, validated_before)
display(trips.data.head())
display(filtered.data.head())
display(report.issues)

,movement_id,user_id,trip_id,movement_seq,mode,purpose,day_type,time_period,user_gender,income_quintile,trip_weight,distance_km,is_peak,mode_sequence,origin_municipality,destination_municipality,origin_longitude,origin_latitude,destination_longitude,destination_latitude,origin_time_utc,destination_time_utc,origin_time_local_hhmm,destination_time_local_hhmm,origin_h3_index,destination_h3_index
0,m0,u0,t0,0,bus,work,weekday,morning,female,3,1.2,5.0,True,bus,Santiago,Providencia,-70.66,-33.45,-70.65,-33.44,2026-01-01 07:10:00+00:00,2026-01-01 07:40:00+00:00,07:10,07:40,88b2c55417fffff,88b2c5541bfffff
1,m1,u1,t1,0,metro,study,weekday,morning,male,2,0.8,12.5,True,metro,Ñuñoa,Santiago,-70.64,-33.46,-70.63,-33.45,2026-01-01 08:00:00+00:00,2026-01-01 08:35:00+00:00,08:00,08:35,88b2c5548dfffff,88b2c554c1fffff
2,m2,u2,t2,0,car,work,weekday,morning,female,5,1.5,1.2,False,car,Puente Alto,Puente Alto,-70.80,-33.60,-70.79,-33.59,2026-01-01 09:30:00+00:00,2026-01-01 10:00:00+00:00,09:30,10:00,88b2c54e11fffff,88b2c54525fffff
3,m3,u3,t3,0,walk,leisure,weekday,morning,male,1,0.7,0.4,False,walk,Santiago,Quilicura,-70.66,-33.45,-70.81,-33.61,2026-01-01 10:00:00+00:00,2026-01-01 10:20:00+00:00,10:00,10:20,88b2c55417fffff,88b2c54e3bfffff
4,m4,u4,t4,0,bus,leisure,weekend,midday,female,4,1.1,25.0,True,bus,Maipú,Maipú,-70.90,-33.70,-70.91,-33.71,2026-01-01 11:00:00+00:00,2026-01-01 11:30:00+00:00,11:00,11:30,88b2c54da1fffff,88b2c0b29bfffff


,movement_id,user_id,trip_id,movement_seq,mode,purpose,day_type,time_period,user_gender,income_quintile,trip_weight,distance_km,is_peak,mode_sequence,origin_municipality,destination_municipality,origin_longitude,origin_latitude,destination_longitude,destination_latitude,origin_time_utc,destination_time_utc,origin_time_local_hhmm,destination_time_local_hhmm,origin_h3_index,destination_h3_index
0,m0,u0,t0,0,bus,work,weekday,morning,female,3,1.2,5.0,True,bus,Santiago,Providencia,-70.66,-33.45,-70.65,-33.44,2026-01-01 07:10:00+00:00,2026-01-01 07:40:00+00:00,07:10,07:40,88b2c55417fffff,88b2c5541bfffff
1,m1,u1,t1,0,metro,study,weekday,morning,male,2,0.8,12.5,True,metro,Ñuñoa,Santiago,-70.64,-33.46,-70.63,-33.45,2026-01-01 08:00:00+00:00,2026-01-01 08:35:00+00:00,08:00,08:35,88b2c5548dfffff,88b2c554c1fffff
3,m3,u3,t3,0,walk,leisure,weekday,morning,male,1,0.7,0.4,False,walk,Santiago,Quilicura,-70.66,-33.45,-70.81,-33.61,2026-01-01 10:00:00+00:00,2026-01-01 10:20:00+00:00,10:00,10:20,88b2c55417fffff,88b2c54e3bfffff
5,m5,u5,t5,0,metro,work,weekday,morning,other,2,1.0,8.2,True,metro,Providencia,Santiago,-70.62,-33.43,-70.63,-33.44,2026-01-01 07:50:00+00:00,2026-01-01 08:10:00+00:00,07:50,08:10,88b2c556bdfffff,88b2c554c9fffff
6,m6,u6,t6,0,bicycle,work,weekday,morning,male,1,0.9,3.0,True,bicycle,Providencia,Ñuñoa,-70.61,-33.42,-70.62,-33.43,2026-01-01 08:20:00+00:00,2026-01-01 08:50:00+00:00,08:20,08:50,88b2c55681fffff,88b2c556bdfffff


[Issue(level='error', code='FLT.TIME.UNSUPPORTED_TIER', message="No se puede aplicar el filtro temporal: el dataset está en 'tier_2' y OP-05 solo evalúa Tier 1 absoluto; la regla se omitirá.", field=None, source_field=None, row_count=None, details={'check': 'time', 'predicate': 'overlaps', 'start': '2026-01-01T07:00:00Z', 'end': '2026-01-01T09:00:00Z', 'temporal_tier': 'tier_2', 'fields_present': ['origin_time_local_hhmm', 'destination_time_local_hhmm'], 'missing_fields': ['origin_time_utc', 'destination_time_utc'], 'expected': 'tier_1 with origin_time_utc and destination_time_utc', 'observed': 'tier_2', 'reason': 'unsupported_temporal_tier', 'action': 'omit_time_filter'}),
 Issue(level='info', code='FLT.INFO.BBOX_APPLIED', message='Se aplicó el filtro bbox y el subconjunto pasó de 12 a 9 filas en este eje.', field=None, source_field=None, row_count=None, details={'check': 'spatial', 'rows_in_scope': 12, 'rows_out_scope': 9, 'rows_removed_scope': 3, 'row_indices_sample_removed': [2, 4,

### Test 5 - metadata / event / summary consistente en fixture validado

Qué prueba: que la operación preserve is_validated, haga append de evento y mantenga consistencia exacta entre report.summary, report.parameters y el evento final.

In [17]:
trips = tripdataset_validated_small()
events_before = deepcopy(trips.metadata["events"])
validated_before = trips.metadata["is_validated"]

options = FilterOptions(
    where={"mode": "bus"},
)

filtered, report = filter_trips(trips, options=options)

assert report.ok is True
assert filtered.metadata["is_validated"] is validated_before
assert len(filtered.metadata["events"]) == len(events_before) + 1

event = get_last_event(filtered)
assert event["op"] == "filter_trips"
assert event["summary"] == report.summary
assert event["parameters"] == report.parameters
assert "issues_summary" in event

# No cambia el historial del input.
assert trips.metadata["events"] == events_before

### Test 6 - caso de política clave: keep_metadata=False

Qué prueba: la política cerrada de metadata mínima cuando keep_metadata=False.

In [18]:
trips = tripdataset_validated_small()
events_before = deepcopy(trips.metadata["events"])

options = FilterOptions(
    where={"mode": ["bus", "metro"]},
    keep_metadata=False,
)

filtered, report = filter_trips(trips, options=options)

assert report.ok is True
assert filtered.data["movement_id"].tolist() == ["m0", "m1", "m4", "m5", "m7"]

# Metadata mínima cerrada por contrato
expected_keys = {"dataset_id", "is_validated", "temporal", "h3", "schema", "domains_effective"}
assert set(filtered.metadata.keys()) == expected_keys
assert filtered.metadata["dataset_id"] == trips.metadata["dataset_id"]
assert filtered.metadata["is_validated"] is True
assert "events" not in filtered.metadata

# El input conserva sus eventos
assert trips.metadata["events"] == events_before

### Test 7 - política clave: strict=True con error recuperable escalar a FilterError

Qué prueba: escalamiento correcto cuando hay issue error recuperable y strict=True.

In [19]:
trips = tripdataset_with_time_space_fields(validated=False, temporal_tier="tier_2")
data_before = trips.data.copy(deep=True)
events_before = deepcopy(trips.metadata["events"])
validated_before = trips.metadata["is_validated"]

raised = None
try:
    filter_trips(
        trips,
        options=FilterOptions(
            time=TimeFilter(
                start="2026-01-01T07:00:00Z",
                end="2026-01-01T09:00:00Z",
                predicate="overlaps",
            ),
            strict=True,
        ),
    )
except Exception as e:
    raised = e

assert raised is not None
assert isinstance(raised, FilterError)
assert raised.issue is not None
assert raised.issue.code == "FLT.TIME.UNSUPPORTED_TIER"

assert_input_not_mutated(trips, data_before, events_before, validated_before)
display(raised.issues)

(Issue(level='error', code='FLT.TIME.UNSUPPORTED_TIER', message="No se puede aplicar el filtro temporal: el dataset está en 'tier_2' y OP-05 solo evalúa Tier 1 absoluto; la regla se omitirá.", field=None, source_field=None, row_count=None, details={'check': 'time', 'predicate': 'overlaps', 'start': '2026-01-01T07:00:00Z', 'end': '2026-01-01T09:00:00Z', 'temporal_tier': 'tier_2', 'fields_present': ['origin_time_local_hhmm', 'destination_time_local_hhmm'], 'missing_fields': ['origin_time_utc', 'destination_time_utc'], 'expected': 'tier_1 with origin_time_utc and destination_time_utc', 'observed': 'tier_2', 'reason': 'unsupported_temporal_tier', 'action': 'omit_time_filter'}),
 Issue(level='info', code='FLT.INFO.FILTER_WITHOUT_EFFECT', message='Los filtros evaluados no produjeron cambios efectivos: no se descartaron filas.', field=None, source_field=None, row_count=None, details={'rows_in': 12, 'rows_out': 12, 'dropped_total': 0, 'filters_requested': ['time'], 'filters_applied': [], 'filt

### Test 8 - caso degradado con warning: resultado vacío retornable

Qué prueba: EMPTY_RESULT como warning retornable, sin abortar.

In [22]:
trips = tripdataset_canonical_small(validated=True)

options = FilterOptions(
    where={"mode": "scooter"},  # valor permitido por dominio, pero ausente en este fixture
)

filtered, report = filter_trips(trips, options=options)

assert filtered.data.empty is True
assert report.ok is True
assert report.summary["rows_in"] == 8
assert report.summary["rows_out"] == 0
assert report.summary["dropped_total"] == 8

assert_codes_present(report, ["FLT.OUTPUT.EMPTY_RESULT"])

event = get_last_event(filtered)
assert event["op"] == "filter_trips"
assert event["summary"] == report.summary
assert filtered.metadata["is_validated"] is True

display(trips.data)
display(filtered.data)
display(report.issues)

,movement_id,user_id,trip_id,movement_seq,mode,purpose,day_type,time_period,user_gender,income_quintile,trip_weight,distance_km,is_peak,mode_sequence,origin_municipality,destination_municipality,origin_longitude,origin_latitude,destination_longitude,destination_latitude,origin_time_utc,destination_time_utc,origin_time_local_hhmm,destination_time_local_hhmm,origin_h3_index,destination_h3_index
0,m0,u0,t0,0,bus,work,weekday,morning,female,3,1.2,5.0,True,bus,Santiago,Providencia,-70.66,-33.45,-70.65,-33.44,2026-01-01 07:10:00+00:00,2026-01-01 07:40:00+00:00,07:10,07:40,88b2c55417fffff,88b2c5541bfffff
1,m1,u1,t1,0,metro,study,weekday,morning,male,2,0.8,12.5,True,metro,Ñuñoa,Santiago,-70.64,-33.46,-70.63,-33.45,2026-01-01 08:00:00+00:00,2026-01-01 08:35:00+00:00,08:00,08:35,88b2c5548dfffff,88b2c554c1fffff
2,m2,u2,t2,0,car,work,weekday,morning,female,5,1.5,1.2,False,car,Puente Alto,Puente Alto,-70.80,-33.60,-70.79,-33.59,2026-01-01 09:30:00+00:00,2026-01-01 10:00:00+00:00,09:30,10:00,88b2c54e11fffff,88b2c54525fffff
3,m3,u3,t3,0,walk,leisure,weekday,morning,male,1,0.7,0.4,False,walk,Santiago,Quilicura,-70.66,-33.45,-70.81,-33.61,2026-01-01 10:00:00+00:00,2026-01-01 10:20:00+00:00,10:00,10:20,88b2c55417fffff,88b2c54e3bfffff
4,m4,u4,t4,0,bus,leisure,weekend,midday,female,4,1.1,25.0,True,bus,Maipú,Maipú,-70.90,-33.70,-70.91,-33.71,2026-01-01 11:00:00+00:00,2026-01-01 11:30:00+00:00,11:00,11:30,88b2c54da1fffff,88b2c0b29bfffff
5,m5,u5,t5,0,metro,work,weekday,morning,other,2,1.0,8.2,True,metro,Providencia,Santiago,-70.62,-33.43,-70.63,-33.44,2026-01-01 07:50:00+00:00,2026-01-01 08:10:00+00:00,07:50,08:10,88b2c556bdfffff,88b2c554c9fffff
6,m6,u6,t6,0,bicycle,work,weekday,morning,male,1,0.9,3.0,True,bicycle,Providencia,Ñuñoa,-70.61,-33.42,-70.62,-33.43,2026-01-01 08:20:00+00:00,2026-01-01 08:50:00+00:00,08:20,08:50,88b2c55681fffff,88b2c556bdfffff
7,m7,u7,t7,0,bus,shopping,weekday,morning,female,2,0.6,6.0,False,bus,Santiago,Santiago,-70.67,-33.46,-70.66,-33.45,2026-01-01 06:30:00+00:00,2026-01-01 06:50:00+00:00,06:30,06:50,88b2c554e5fffff,88b2c55417fffff


,movement_id,user_id,trip_id,movement_seq,mode,purpose,day_type,time_period,user_gender,income_quintile,trip_weight,distance_km,is_peak,mode_sequence,origin_municipality,destination_municipality,origin_longitude,origin_latitude,destination_longitude,destination_latitude,origin_time_utc,destination_time_utc,origin_time_local_hhmm,destination_time_local_hhmm,origin_h3_index,destination_h3_index


[Issue(level='info', code='FLT.INFO.WHERE_APPLIED', message='Se aplicó la cláusula where y el subconjunto pasó de 8 a 0 filas en este eje.', field=None, source_field=None, row_count=None, details={'check': 'where', 'where_fields': ['mode'], 'rows_in_scope': 8, 'rows_out_scope': 0, 'rows_removed_scope': 8, 'row_indices_sample_removed': [0, 1, 2, 3, 4, 5, 6, 7], 'movement_ids_sample_removed': ['m0', 'm1', 'm2', 'm3', 'm4', 'm5', 'm6', 'm7'], 'values_sample': [{'mode': 'bus'}, {'mode': 'metro'}, {'mode': 'car'}, {'mode': 'walk'}, {'mode': 'bus'}, {'mode': 'metro'}, {'mode': 'bicycle'}, {'mode': 'bus'}], 'policy': 'and_between_fields_and_ops'}),
 Issue(level='warning', code='FLT.OUTPUT.EMPTY_RESULT', message='El filtrado produjo un dataset vacío.', field=None, source_field=None, row_count=None, details={'rows_in': 8, 'rows_out': 0, 'dropped_total': 8, 'filters_requested': ['where'], 'filters_applied': ['where'], 'filters_omitted': [], 'dropped_by_filter': {'where': 8, 'time': 0, 'bbox': 0,

### Test 9 - truncamiento de issues + bloque limits + consistencia evento/report

Qué prueba: control de tamaño del reporte y consistencia del bloque limits.

In [23]:
trips = tripdataset_with_time_space_fields(validated=False, temporal_tier="tier_2")

options = FilterOptions(
    where={
        "does_not_exist": "x",            # FIELD_NOT_FOUND
        "mode": {"gt": "bus"},            # OP_INCOMPATIBLE
        "purpose": {"in": set(["work"])}, # INVALID_VALUE_SHAPE
    },
    time=TimeFilter(
        start="2026-01-01T07:00:00Z",
        end="2026-01-01T09:00:00Z",
        predicate="overlaps",             # UNSUPPORTED_TIER
    ),
    strict=False,
)

filtered, report = filter_trips(
    trips,
    options=options,
    max_issues=2,
)

codes = [issue.code for issue in report.issues]
assert "FLT.LIMIT.ISSUES_TRUNCATED" in codes

assert "limits" in report.summary
assert report.summary["limits"]["issues_truncated"] is True
assert report.summary["limits"]["max_issues"] == 2
assert report.summary["limits"]["n_issues_emitted"] <= 2
assert report.summary["limits"]["n_issues_detected_total"] >= report.summary["limits"]["n_issues_emitted"]

event = get_last_event(filtered)
assert event["summary"] == report.summary
assert event["parameters"] == report.parameters
assert "issues_summary" in event